In [1]:
import os
import json
import pandas as pd
import numpy as np
import shutil
# import matplotlib.pyplot as plt

## 1. Set the rank to pick the best runs to evaluate the test set on

In [19]:
files = os.listdir("/home/jayje/projects/aip-craffel/moose/lora_training")

cols = ['run_name','dataset_name','task','rank','lr','warmup_ratio','epoch','step',
        'batch_size','effective_batch_size','max_seq_len','seed',
        'data_size','valid/exact_string_match_accuracy','final_validation_loss','time_taken']

ls = []
for file_path in files:
    file_name = f"/home/jayje/projects/aip-craffel/moose/lora_training/{file_path}/job_summary.json"
    if os.path.exists(file_name):
        with open(file_name, "r") as f:
            job = json.load(f)
        temp = pd.DataFrame.from_dict(job, orient='index').T
        temp_reduced = temp[cols]
        ls.append(temp_reduced)

df = pd.concat(ls)
df.head()

,run_name,dataset_name,task,rank,lr,warmup_ratio,epoch,step,batch_size,effective_batch_size,max_seq_len,seed,data_size,valid/exact_string_match_accuracy,final_validation_loss,time_taken
0,lora_baseline_task10_lr1e-4_step100_rank16,masakhane/masakhanews,swa,16,0.0001,0.1,40,100,1,8,2048,123,100,0.866667,inf,0.151352
0,lora_baseline_task10_lr1e-4_step100_rank32,masakhane/masakhanews,swa,32,0.0001,0.1,40,100,1,8,2048,123,100,0.8,inf,0.159346
0,lora_baseline_task10_lr1e-4_step100_rank64,masakhane/masakhanews,swa,64,0.0001,0.1,40,100,1,8,2048,123,100,0.8,inf,0.166236
0,lora_baseline_task10_lr1e-4_step400_rank16,masakhane/masakhanews,swa,16,0.0001,0.1,40,400,1,8,2048,123,100,0.733333,inf,0.162731
0,lora_baseline_task10_lr1e-4_step400_rank32,masakhane/masakhanews,swa,32,0.0001,0.1,40,400,1,8,2048,123,100,0.8,inf,0.164933


In [20]:
df['val_accuracy_rank'] = df.groupby(['dataset_name','task'])['valid/exact_string_match_accuracy'].rank(method='dense', ascending=False)
df['val_loss_rank'] = df.groupby(['dataset_name','task'])['final_validation_loss'].rank(method='dense', ascending=True)
df['rank_summed'] = df['val_accuracy_rank'] + df['val_loss_rank']

In [21]:
df['task_num']=df['run_name'].str.split('baseline_').str[-1].str.split('_lr').str[0]

In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1050 entries, 0 to 0
Data columns (total 20 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   run_name                           1050 non-null   object 
 1   dataset_name                       1050 non-null   object 
 2   task                               1050 non-null   object 
 3   rank                               1050 non-null   object 
 4   lr                                 1050 non-null   object 
 5   warmup_ratio                       1050 non-null   object 
 6   epoch                              1050 non-null   object 
 7   step                               1050 non-null   object 
 8   batch_size                         1050 non-null   object 
 9   effective_batch_size               1050 non-null   object 
 10  max_seq_len                        1050 non-null   object 
 11  seed                               1050 non-null   object 
 12  

In [23]:
total_count = df.groupby(['dataset_name','task']).agg({'seed':'count'}).rename(columns={'seed':'num_runs'}).reset_index()

In [24]:
df.shape

(1050, 20)

In [25]:
df = pd.merge(
    df,
    total_count,
    on = ['dataset_name','task'],
    how='inner'
)

In [26]:
df.shape

(1050, 21)

In [27]:
df.head()

,run_name,dataset_name,task,rank,lr,warmup_ratio,epoch,step,batch_size,effective_batch_size,...,seed,data_size,valid/exact_string_match_accuracy,final_validation_loss,time_taken,val_accuracy_rank,val_loss_rank,rank_summed,task_num,num_runs
0,lora_baseline_task10_lr1e-4_step100_rank16,masakhane/masakhanews,swa,16,0.0001,0.1,40,100,1,8,...,123,100,0.866667,inf,0.151352,1.0,1.0,2.0,task10,12
1,lora_baseline_task10_lr1e-4_step100_rank32,masakhane/masakhanews,swa,32,0.0001,0.1,40,100,1,8,...,123,100,0.8,inf,0.159346,2.0,1.0,3.0,task10,12
2,lora_baseline_task10_lr1e-4_step100_rank64,masakhane/masakhanews,swa,64,0.0001,0.1,40,100,1,8,...,123,100,0.8,inf,0.166236,2.0,1.0,3.0,task10,12
3,lora_baseline_task10_lr1e-4_step400_rank16,masakhane/masakhanews,swa,16,0.0001,0.1,40,400,1,8,...,123,100,0.733333,inf,0.162731,3.0,1.0,4.0,task10,12
4,lora_baseline_task10_lr1e-4_step400_rank32,masakhane/masakhanews,swa,32,0.0001,0.1,40,400,1,8,...,123,100,0.8,inf,0.164933,2.0,1.0,3.0,task10,12


In [33]:
df['task_num'].nunique() # 10 or so tasks failed to generate answer because the training size too large

52

In [32]:
df['num_runs'].value_counts()

num_runs
24    528
18    144
23     92
17     85
15     75
16     64
20     20
19     19
12     12
11     11
Name: count, dtype: int64

In [34]:
df[df['num_runs']==24]['task_num'].nunique() # JJE: should past this test

22

In [36]:
df_fin = df[df['num_runs']==24].reset_index(drop=True)

### Ranked accuracy

In [37]:
df_acc_rank = df_fin[(df_fin['val_accuracy_rank']==1)].reset_index(drop=True)
df_acc_rank['combined_rank'] = df_acc_rank.groupby(['dataset_name','task'])['rank_summed'].rank(method='dense', ascending=True)
df_acc_rank[df_acc_rank['combined_rank']==1]['rank_summed'].value_counts()

rank_summed
2.0    14
3.0     3
6.0     2
4.0     2
8.0     1
Name: count, dtype: int64

In [38]:
print(df_acc_rank[df_acc_rank['combined_rank']==1].shape) # jje: total count should be ~ 62
df_acc_rank[df_acc_rank['combined_rank']==1].head() 

(22, 22)


,run_name,dataset_name,task,rank,lr,warmup_ratio,epoch,step,batch_size,effective_batch_size,...,data_size,valid/exact_string_match_accuracy,final_validation_loss,time_taken,val_accuracy_rank,val_loss_rank,rank_summed,task_num,num_runs,combined_rank
0,lora_baseline_task14_lr3e-4_step400_rank32,cardiffnlp/tweet_topic_single,None,32,0.0003,0.1,40,400,1,8,...,100,0.9,0.156266,0.145842,1.0,1.0,2.0,task14,24,1.0
11,lora_baseline_task15_lr3e-4_step400_rank16,jpwahle/machine-paraphrase-dataset,None,16,0.0003,0.1,40,400,1,8,...,100,1.0,0.000001,0.434782,1.0,1.0,2.0,task15,24,1.0
22,lora_baseline_task17_lr3e-4_step400_rank32,facebook/anli,None,32,0.0003,0.1,40,400,1,8,...,100,0.75,0.272094,0.093209,1.0,5.0,6.0,task17,24,1.0
23,lora_baseline_task18_lr1e-4_step400_rank16,google-research-datasets/circa,None,16,0.0001,0.1,40,400,1,8,...,100,0.85,0.171218,0.116554,1.0,2.0,3.0,task18,24,1.0
29,lora_baseline_task19_lr3e-4_step100_rank32,cais/mmlu,None,32,0.0003,0.1,40,100,1,8,...,100,0.7,0.253853,0.080687,1.0,1.0,2.0,task19,24,1.0


In [39]:
df_acc_rank[df_acc_rank['combined_rank']==1].to_csv("../weight_processing/lora_ft_highest_val_acc_run.csv", index=False)

### Ranked loss

In [40]:
df_loss_rank = df_fin[(df_fin['val_loss_rank']==1)].reset_index(drop=True)
df_loss_rank['combined_rank'] = df_loss_rank.groupby(['dataset_name','task'])['rank_summed'].rank(method='dense', ascending=True)
df_loss_rank[df_loss_rank['combined_rank']==1]['rank_summed'].value_counts()

rank_summed
2.0    14
3.0     6
4.0     1
6.0     1
Name: count, dtype: int64

In [41]:
df_loss_rank[(df_loss_rank['combined_rank']==1)].to_csv("../weight_processing/lora_ft_lowest_val_loss_run.csv", index=False)

In [42]:
print(df_loss_rank[(df_loss_rank['combined_rank']==1)].shape) # jje: total count should be ~ 62
df_loss_rank[(df_loss_rank['combined_rank']==1)].head()

(22, 22)


,run_name,dataset_name,task,rank,lr,warmup_ratio,epoch,step,batch_size,effective_batch_size,...,data_size,valid/exact_string_match_accuracy,final_validation_loss,time_taken,val_accuracy_rank,val_loss_rank,rank_summed,task_num,num_runs,combined_rank
0,lora_baseline_task14_lr3e-4_step400_rank32,cardiffnlp/tweet_topic_single,None,32,0.0003,0.1,40,400,1,8,...,100,0.9,0.156266,0.145842,1.0,1.0,2.0,task14,24,1.0
1,lora_baseline_task15_lr3e-4_step400_rank16,jpwahle/machine-paraphrase-dataset,None,16,0.0003,0.1,40,400,1,8,...,100,1.0,0.000001,0.434782,1.0,1.0,2.0,task15,24,1.0
2,lora_baseline_task17_lr3e-4_step100_rank32,facebook/anli,None,32,0.0003,0.1,40,100,1,8,...,100,0.65,0.241553,0.07247,3.0,1.0,4.0,task17,24,1.0
3,lora_baseline_task18_lr3e-4_step400_rank32,google-research-datasets/circa,None,32,0.0003,0.1,40,400,1,8,...,100,0.8,0.134973,0.131154,2.0,1.0,3.0,task18,24,1.0
4,lora_baseline_task19_lr3e-4_step100_rank32,cais/mmlu,None,32,0.0003,0.1,40,100,1,8,...,100,0.7,0.253853,0.080687,1.0,1.0,2.0,task19,24,1.0


## 2. Pick the lora evaluation metric based on test performance

In [65]:
# Read test-set accuracy for the LoRA models
# jje: change the lora_training -> lora_eval

files = os.listdir("/home/jayje/projects/aip-craffel/moose/lora_training")

In [87]:
ls = []
for file_path in files:
    file_name = f"/home/jayje/projects/aip-craffel/moose/lora_training/{file_path}/job_summary.json"
    if os.path.exists(file_name):
        with open(file_name, "r") as f:
            job = json.load(f)
        temp = pd.DataFrame.from_dict(job, orient='index').T
        temp_reduced = temp
        ls.append(temp_reduced)

df_test = pd.concat(ls)
df_test.head()

,run_name,output_dir,model_type,compression_weight_dir,rank,router_activation,router_base_as_an_expert,expert_sigma_normalization,lr,warmup_ratio,...,lr_multiplier_routing,lr_multiplier_else,num_experts,calculate_all_metrics,save_model,force_routing,single_sigma,valid/exact_string_match_accuracy,final_validation_loss,time_taken
0,lora_baseline_task10_lr5e-5_step100_rank16,/home/jayje/projects/aip-craffel/moose/lora_tr...,lora,outputs/global_jd_moose_512,16,linear,False,False,0.00005,0.1,...,1.0,0.0,-1,False,True,None,False,0.8,inf,0.147491
0,lora_baseline_task10_lr5e-5_step100_rank32,/home/jayje/projects/aip-craffel/moose/lora_tr...,lora,outputs/global_jd_moose_512,32,linear,False,False,0.00005,0.1,...,1.0,0.0,-1,False,True,None,False,0.733333,inf,0.150174
0,lora_baseline_task10_lr5e-5_step100_rank64,/home/jayje/projects/aip-craffel/moose/lora_tr...,lora,outputs/global_jd_moose_512,64,linear,False,False,0.00005,0.1,...,1.0,0.0,-1,False,True,None,False,0.666667,inf,0.157341
0,lora_baseline_task11_lr5e-5_step100_rank16,/home/jayje/projects/aip-craffel/moose/lora_tr...,lora,outputs/global_jd_moose_512,16,linear,False,False,0.00005,0.1,...,1.0,0.0,-1,False,True,None,False,1.0,inf,0.118914
0,lora_baseline_task11_lr5e-5_step100_rank64,/home/jayje/projects/aip-craffel/moose/lora_tr...,lora,outputs/global_jd_moose_512,64,linear,False,False,0.00005,0.1,...,1.0,0.0,-1,False,True,None,False,1.0,inf,0.128676


In [88]:
best_acc_runs = pd.read_csv("../weight_processing/lora_ft_highest_val_acc_run.csv")
best_loss_runs = pd.read_csv("../weight_processing/lora_ft_lowest_val_loss_run.csv")

In [89]:
best_acc_runs['task_num']=best_acc_runs['run_name'].str.split('baseline_').str[-1].str.split('_lr').str[0]
best_loss_runs['task_num']=best_loss_runs['run_name'].str.split('baseline_').str[-1].str.split('_lr').str[0]

best_acc_runs['source']='val_acc'
best_loss_runs['source']='val_loss'

In [90]:
print(best_acc_runs['task_num'].nunique())
print(best_loss_runs['task_num'].nunique())

52
52


In [91]:
best_acc_runs['task'] = best_acc_runs['task'].fillna("None")
best_loss_runs['task'] = best_loss_runs['task'].fillna("None")

In [92]:
subset_cols = ['run_name','dataset_name','task','task_num','source','val_accuracy_rank','val_loss_rank','combined_rank','rank_summed']
best_runs = pd.concat([best_acc_runs, best_loss_runs])[subset_cols]
print(best_acc_runs['task'].nunique())
print(best_runs['task'].nunique())
print(best_runs['task'].shape)

37
37
(106,)


In [93]:
print(best_runs[best_runs['rank_summed']==2].sort_values('run_name')['run_name'].nunique())
print(best_runs[best_runs['rank_summed']!=2].sort_values('run_name')['run_name'].nunique()) # 37 + 30

35
36


In [94]:
# jje: fix this part <- exclude this later!
df_test['test/exact_string_match_accuracy'] = df_test['valid/exact_string_match_accuracy'].apply(lambda x: x + np.random.randn()) 

In [95]:
best_runs = pd.merge(
    best_runs,
    df_test[['run_name','test/exact_string_match_accuracy']],
    on = ['run_name'],
    how = 'left'
)

In [96]:
best_runs['test_acc_rank'] = best_runs.groupby(['dataset_name','task'])['test/exact_string_match_accuracy'].rank(method='dense', ascending=False)

In [97]:
best_run_res = pd.pivot(
    best_runs,
    index='task_num',
    columns='source',
    values='test/exact_string_match_accuracy'
)

ValueError: Index contains duplicate entries, cannot reshape

In [78]:
best_run_res.shape

(52, 2)

In [79]:
print(best_run_res[best_run_res['val_acc'] > best_run_res['val_loss']].shape[0])
print(best_run_res[best_run_res['val_acc'] == best_run_res['val_loss']].shape[0])
print(best_run_res[best_run_res['val_acc'] < best_run_res['val_loss']].shape[0])

8
37
7


In [80]:
best_run_res

source,val_acc,val_loss
task_num,,
task10,0.594355,0.594355
task11,2.338752,2.338752
task13,0.901180,0.901180
task14,1.377360,1.377360
task15,2.599061,2.599061
task17,1.198876,0.738511
task18,0.264042,0.264042
task19,1.800203,1.800203
task20,0.845412,0.156745


## 3. Based on the better metric, move the best loras into one folder

    1. get the complete run result w/ hyperparam + test acc and
    2. upload the models to the correct folder

In [81]:
files = os.listdir("/home/jayje/projects/aip-craffel/moose/lora_training")

cols = ['run_name','dataset_name','task','rank','lr','warmup_ratio','epoch','step',
        'batch_size','effective_batch_size','max_seq_len','seed',
        'data_size','valid/exact_string_match_accuracy','final_validation_loss','time_taken']

ls = []
for file_path in files:
    file_name = f"/home/jayje/projects/aip-craffel/moose/lora_training/{file_path}/job_summary.json"
    if os.path.exists(file_name):
        with open(file_name, "r") as f:
            job = json.load(f)
        temp = pd.DataFrame.from_dict(job, orient='index').T
        temp_reduced = temp[cols]
        ls.append(temp_reduced)

df_train = pd.concat(ls)
df_train.head()

,run_name,dataset_name,task,rank,lr,warmup_ratio,epoch,step,batch_size,effective_batch_size,max_seq_len,seed,data_size,valid/exact_string_match_accuracy,final_validation_loss,time_taken
0,lora_baseline_task10_lr5e-5_step100_rank16,masakhane/masakhanews,swa,16,0.00005,0.1,40,100,1,8,2048,123,100,0.8,inf,0.147491
0,lora_baseline_task10_lr5e-5_step100_rank32,masakhane/masakhanews,swa,32,0.00005,0.1,40,100,1,8,2048,123,100,0.733333,inf,0.150174
0,lora_baseline_task10_lr5e-5_step100_rank64,masakhane/masakhanews,swa,64,0.00005,0.1,40,100,1,8,2048,123,100,0.666667,inf,0.157341
0,lora_baseline_task11_lr5e-5_step100_rank16,masakhane/masakhanews,yor,16,0.00005,0.1,40,100,1,8,2048,123,100,1.0,inf,0.118914
0,lora_baseline_task11_lr5e-5_step100_rank64,masakhane/masakhanews,yor,64,0.00005,0.1,40,100,1,8,2048,123,100,1.0,inf,0.128676


In [ ]:
df_train = pd.merge(
    df_train,
    df_test[['run_name','test/exact_string_match_accuracy']],
    on = ['run_name'],
    how='left'    
)

In [20]:
df_train['best_model'] = df_train.groupby(['dataset_name','task'])['final_validation_loss'].rank(ascending=True)

In [23]:
df_train[df_train['best_model']==1]

,run_name,dataset_name,task,rank,lr,warmup_ratio,epoch,step,batch_size,effective_batch_size,max_seq_len,seed,data_size,valid/exact_string_match_accuracy,final_validation_loss,time_taken,best_model
0,lora_baseline_task13_lr5e-5_step100_rank64,community-datasets/disaster_response_messages,None,64,0.00005,0.1,40,100,1,8,2048,123,100,0.8,0.152417,0.102094,1.0
0,lora_baseline_task14_lr5e-5_step100_rank16,cardiffnlp/tweet_topic_single,None,16,0.00005,0.1,40,100,1,8,2048,123,100,0.75,0.31204,0.065449,1.0
0,lora_baseline_task15_lr5e-5_step400_rank16,jpwahle/machine-paraphrase-dataset,None,16,0.00005,0.1,40,400,1,8,2048,123,100,1.0,0.000005,0.443552,1.0
0,lora_baseline_task17_lr5e-5_step100_rank32,facebook/anli,None,32,0.00005,0.1,40,100,1,8,2048,123,100,0.55,0.300221,0.073468,1.0
0,lora_baseline_task18_lr5e-5_step400_rank32,google-research-datasets/circa,None,32,0.00005,0.1,40,400,1,8,2048,123,100,0.8,0.188481,0.133362,1.0
0,lora_baseline_task19_lr5e-5_step100_rank128,cais/mmlu,None,128,0.00005,0.1,40,100,1,8,2048,123,100,0.5,0.278418,0.092523,1.0
0,lora_baseline_task20_lr5e-5_step400_rank16,TIGER-Lab/MMLU-Pro,None,16,0.00005,0.1,40,400,1,8,2048,123,100,0.25,0.7133,0.095997,1.0
0,lora_baseline_task21_lr5e-5_step100_rank64,deepmind/aqua_rat,tokenized,64,0.00005,0.1,40,100,1,8,2048,123,100,0.25,0.533477,0.100559,1.0
0,lora_baseline_task22_lr5e-5_step100_rank64,tau/commonsense_qa,None,64,0.00005,0.1,40,100,1,8,2048,123,100,0.55,0.338222,0.070935,1.0
0,lora_baseline_task23_lr5e-5_step100_rank64,nvidia/OpenMathInstruct-2,None,64,0.00005,0.1,40,100,1,8,2048,123,100,0.05,1.122841,0.601085,1.0


In [30]:
def to_comma_name(dataset_name: str, task: str | None) -> str:
    if task is None:
        task = "None"
    return f"{dataset_name.replace('/', ',')},,{task.replace('/', ',')}"

In [31]:
df_train['dir_name'] = df_train.apply(lambda x: to_comma_name(x['dataset_name'], x['task']), axis=1)

In [46]:
## run_name
best_run_path_dir = zip(df_train[df_train['best_model']==1]['run_name'],df_train[df_train['best_model']==1]['dir_name'])
for run, dir_name in best_run_path_dir:    
    src=f"/home/jayje/projects/aip-craffel/moose/lora_training/{run}"
    dst=f"/home/jayje/projects/aip-craffel/moose/best_lora/{dir_name}"
    print(src)
    print(dst)
    print("\n")

/home/jayje/projects/aip-craffel/moose/lora_training/lora_baseline_task13_lr5e-5_step100_rank64
/home/jayje/projects/aip-craffel/moose/best_lora/community-datasets,disaster_response_messages,,None


/home/jayje/projects/aip-craffel/moose/lora_training/lora_baseline_task14_lr5e-5_step100_rank16
/home/jayje/projects/aip-craffel/moose/best_lora/cardiffnlp,tweet_topic_single,,None


/home/jayje/projects/aip-craffel/moose/lora_training/lora_baseline_task15_lr5e-5_step400_rank16
/home/jayje/projects/aip-craffel/moose/best_lora/jpwahle,machine-paraphrase-dataset,,None


/home/jayje/projects/aip-craffel/moose/lora_training/lora_baseline_task17_lr5e-5_step100_rank32
/home/jayje/projects/aip-craffel/moose/best_lora/facebook,anli,,None


/home/jayje/projects/aip-craffel/moose/lora_training/lora_baseline_task18_lr5e-5_step400_rank32
/home/jayje/projects/aip-craffel/moose/best_lora/google-research-datasets,circa,,None


/home/jayje/projects/aip-craffel/moose/lora_training/lora_baseline_task19_lr5e-

Once the above is done, training and selecting the best LoRA is done.

## Scratch -- Two files

In [113]:
files1 = os.listdir("/home/jayje/projects/aip-craffel/moose/lora_training")
files2 = os.listdir("/home/jayje/projects/aip-craffel/moose/lora_training_bs4")

In [114]:
cols = ['rank','lr','warmup_ratio','epoch','step','dataset_name','task',
        'batch_size','effective_batch_size','max_seq_len','seed',
        'data_size','valid/exact_string_match_accuracy','final_validation_loss','time_taken']

In [115]:
ls1 = []
for file_path in files1:
    file_name = f"/home/jayje/projects/aip-craffel/moose/lora_training/{file_path}/job_summary.json"
    if os.path.exists(file_name):
        with open(file_name, "r") as f:
            job = json.load(f)
        temp = pd.DataFrame.from_dict(job, orient='index').T[cols]
        ls1.append(temp)

ls2 = []
for file_path in files2:
    file_name = f"/home/jayje/projects/aip-craffel/moose/lora_training_bs4/{file_path}/job_summary.json"
    if os.path.exists(file_name):
        if os.path.isfile(file_name):
            with open(file_name, "r") as f:
                job = json.load(f)
            temp = pd.DataFrame.from_dict(job, orient='index').T[cols]
            ls2.append(temp)
        else:
            print(f"{file_name} is not a file!")

/home/jayje/projects/aip-craffel/moose/lora_training_bs4/lora_baseline_task0_lr5e-5_step100_rank16/job_summary.json is not a file!


In [116]:
df1 = pd.concat(ls1)
df1.head()

,rank,lr,warmup_ratio,epoch,step,dataset_name,task,batch_size,effective_batch_size,max_seq_len,seed,data_size,valid/exact_string_match_accuracy,final_validation_loss,time_taken
0,16,0.00005,0.1,40,100,masakhane/masakhanews,swa,1,8,2048,123,100,0.8,inf,0.147491
0,16,0.00005,0.1,40,100,masakhane/masakhanews,yor,1,8,2048,123,100,1.0,inf,0.118914
0,16,0.00005,0.1,40,100,cardiffnlp/tweet_topic_single,None,1,8,2048,123,100,0.75,0.31204,0.065449
0,32,0.00005,0.1,40,100,cardiffnlp/tweet_topic_single,None,1,8,2048,123,100,0.75,0.315016,0.067938
0,16,0.00005,0.1,40,100,jpwahle/machine-paraphrase-dataset,None,1,8,2048,123,100,1.0,0.000013,0.264802


In [117]:
df2 = pd.concat(ls2)
df2.head()

,rank,lr,warmup_ratio,epoch,step,dataset_name,task,batch_size,effective_batch_size,max_seq_len,seed,data_size,valid/exact_string_match_accuracy,final_validation_loss,time_taken
0,128,0.0001,0.1,40,100,community-datasets/disaster_response_messages,None,4,8,2048,123,100,0.8,0.136352,0.0721
0,128,0.00005,0.1,40,100,community-datasets/disaster_response_messages,None,4,8,2048,123,100,0.8,0.153395,0.072184
0,16,0.00005,0.1,40,100,community-datasets/disaster_response_messages,None,4,8,2048,123,100,0.8,0.166136,0.063025
0,32,0.00005,0.1,40,100,community-datasets/disaster_response_messages,None,4,8,2048,123,100,0.8,0.14919,0.063512
0,64,0.00005,0.1,40,100,community-datasets/disaster_response_messages,None,4,8,2048,123,100,0.8,0.14979,0.066423


In [118]:
df1['val_accuracy_rank'] = df1.groupby(['dataset_name','task'])['valid/exact_string_match_accuracy'].rank(method='dense', ascending=False)
df1['val_loss_rank'] = df1.groupby(['dataset_name','task'])['final_validation_loss'].rank(method='dense', ascending=True)

In [119]:
df2['val_accuracy_rank'] = df2.groupby(['dataset_name','task'])['valid/exact_string_match_accuracy'].rank(method='dense', ascending=False)
df2['val_loss_rank'] = df2.groupby(['dataset_name','task'])['final_validation_loss'].rank(method='dense', ascending=True)

In [120]:
df1.head()

,rank,lr,warmup_ratio,epoch,step,dataset_name,task,batch_size,effective_batch_size,max_seq_len,seed,data_size,valid/exact_string_match_accuracy,final_validation_loss,time_taken,val_accuracy_rank,val_loss_rank
0,16,0.00005,0.1,40,100,masakhane/masakhanews,swa,1,8,2048,123,100,0.8,inf,0.147491,1.0,1.0
0,16,0.00005,0.1,40,100,masakhane/masakhanews,yor,1,8,2048,123,100,1.0,inf,0.118914,1.0,1.0
0,16,0.00005,0.1,40,100,cardiffnlp/tweet_topic_single,None,1,8,2048,123,100,0.75,0.31204,0.065449,1.0,1.0
0,32,0.00005,0.1,40,100,cardiffnlp/tweet_topic_single,None,1,8,2048,123,100,0.75,0.315016,0.067938,1.0,2.0
0,16,0.00005,0.1,40,100,jpwahle/machine-paraphrase-dataset,None,1,8,2048,123,100,1.0,0.000013,0.264802,1.0,1.0


In [121]:
df1[(df1['val_accuracy_rank']==1) & (df1['task']=='boolean_expressions')]

,rank,lr,warmup_ratio,epoch,step,dataset_name,task,batch_size,effective_batch_size,max_seq_len,seed,data_size,valid/exact_string_match_accuracy,final_validation_loss,time_taken,val_accuracy_rank,val_loss_rank
0,16,0.00005,0.1,40,100,data/boolean_expressions.json,boolean_expressions,1,8,2048,123,100,0.75,0.137722,0.062469,1.0,2.0
0,32,0.00005,0.1,40,100,data/boolean_expressions.json,boolean_expressions,1,8,2048,123,100,0.75,0.135754,0.065205,1.0,1.0
0,64,0.00005,0.1,40,100,data/boolean_expressions.json,boolean_expressions,1,8,2048,123,100,0.75,0.141319,0.067776,1.0,3.0


In [122]:
df2[(df2['val_accuracy_rank']==1) & (df2['task']=='boolean_expressions')]

,rank,lr,warmup_ratio,epoch,step,dataset_name,task,batch_size,effective_batch_size,max_seq_len,seed,data_size,valid/exact_string_match_accuracy,final_validation_loss,time_taken,val_accuracy_rank,val_loss_rank
0,16,0.0001,0.1,40,400,data/boolean_expressions.json,boolean_expressions,4,8,2048,123,100,0.9,0.121012,0.029389,1.0,5.0
0,16,0.00005,0.1,40,400,data/boolean_expressions.json,boolean_expressions,4,8,2048,123,100,0.9,0.120395,0.033623,1.0,4.0
0,32,0.00005,0.1,40,400,data/boolean_expressions.json,boolean_expressions,4,8,2048,123,100,0.9,0.117619,0.036251,1.0,3.0
0,64,0.00005,0.1,40,400,data/boolean_expressions.json,boolean_expressions,4,8,2048,123,100,0.9,0.114556,0.041315,1.0,2.0


In [123]:
df2.shape

(473, 17)

In [124]:
hyperparam_cols = ['rank','lr','step']
task_cols = ['dataset_name','task']
metric_cols = ['valid/exact_string_match_accuracy', 'final_validation_loss','val_accuracy_rank','val_loss_rank']
df_merge = pd.merge(
    df1[hyperparam_cols + task_cols + metric_cols],
    df2[df2['val_loss_rank']==1].drop_duplicates(subset=['rank','lr','step','dataset_name','task'])[hyperparam_cols + task_cols + metric_cols],
    #df2[hyperparam_cols + task_cols + metric_cols],
    on = hyperparam_cols + task_cols,
    how = 'inner',
    suffixes = ['_bs1','_bs4']
)

In [125]:
examine_cols = [
         'dataset_name','task',
         'valid/exact_string_match_accuracy_bs1','valid/exact_string_match_accuracy_bs4',
         'final_validation_loss_bs1', 'final_validation_loss_bs4',
         'val_accuracy_rank_bs1', 'val_accuracy_rank_bs4',
         'val_loss_rank_bs1', 'val_loss_rank_bs4'
         ]

In [126]:
df_merge

,rank,lr,step,dataset_name,task,valid/exact_string_match_accuracy_bs1,final_validation_loss_bs1,val_accuracy_rank_bs1,val_loss_rank_bs1,valid/exact_string_match_accuracy_bs4,final_validation_loss_bs4,val_accuracy_rank_bs4,val_loss_rank_bs4
0,16,0.00005,100,cais/mmlu,None,0.5,0.304563,1.0,1.0,0.45,0.305719,3.0,1.0
1,16,0.00005,100,nyu-mll/glue,cola,0.85,0.060789,1.0,1.0,0.85,0.060113,1.0,1.0
2,16,0.00005,100,data/formal_fallacies_syllogisms_negation.json,formal_fallacies_syllogisms_negation,0.5,0.22848,1.0,1.0,0.55,0.226888,2.0,1.0


In [105]:
print(df_merge[df_merge['valid/exact_string_match_accuracy_bs1'] < df_merge['valid/exact_string_match_accuracy_bs4']].shape[0]/df_merge.shape[0])
print(df_merge[df_merge['valid/exact_string_match_accuracy_bs1'] > df_merge['valid/exact_string_match_accuracy_bs4']].shape[0]/df_merge.shape[0])
print(df_merge[df_merge['valid/exact_string_match_accuracy_bs1'] == df_merge['valid/exact_string_match_accuracy_bs4']].shape[0]/df_merge.shape[0])

print(df_merge[df_merge['valid/exact_string_match_accuracy_bs1'] < df_merge['valid/exact_string_match_accuracy_bs4']].shape[0])
print(df_merge[df_merge['valid/exact_string_match_accuracy_bs1'] > df_merge['valid/exact_string_match_accuracy_bs4']].shape[0])
print(df_merge[df_merge['valid/exact_string_match_accuracy_bs1'] == df_merge['valid/exact_string_match_accuracy_bs4']].shape[0])


0.020833333333333332
0.0
0.020833333333333332
1
0
1


In [92]:
print(df_merge[df_merge['val_loss_rank_bs1'] < df_merge['val_loss_rank_bs4']].shape[0]/df_merge.shape[0])
print(df_merge[df_merge['val_loss_rank_bs1'] > df_merge['val_loss_rank_bs4']].shape[0]/df_merge.shape[0])
print(df_merge[df_merge['val_loss_rank_bs1'] == df_merge['val_loss_rank_bs4']].shape[0]/df_merge.shape[0])


print(df_merge[df_merge['val_loss_rank_bs1'] < df_merge['val_loss_rank_bs4']].shape[0])
print(df_merge[df_merge['val_loss_rank_bs1'] > df_merge['val_loss_rank_bs4']].shape[0])
print(df_merge[df_merge['val_loss_rank_bs1'] == df_merge['val_loss_rank_bs4']].shape[0])

0.8958333333333334
0.0
0.0625
43
0
3
